# 03 — LLM Regression Testing Harness

This notebook accompanies `06-development-challenge-llm-evaluation-before-production.md` and
`03-production-challenge-observability-incident-response.md`.

It builds a small **golden test set** of prompt/expected-behavior pairs — the pre-launch evaluation
asset described in Chapter 06 — and a harness that runs two **mocked "prompt versions"** against it,
flagging which golden cases **regressed** between versions. This is the offline, fully-scriptable
shape of the pre-deploy gate referenced in Chapter 03: before a prompt/model/config change ships,
run it against the golden set and block (or flag for review) any regression, rather than discovering
the regression from live traffic after the fact.

Everything here is pure Python (`unittest.mock`-style fakes, no real LLM calls, no network, no
credentials) — the "LLM" is a small deterministic mock function standing in for a real model call.

## 1. The golden test set

Each golden case has:
- an `id` and a `prompt` (the input),
- a `category` (mirrors Chapter 06's evaluation categories: groundedness, robustness/rephrasing,
  refusal-when-unsupported, and general quality),
- a scoring `rubric` expressed as simple, checkable predicates rather than a single exact-match
  string — because generative outputs are rarely string-matchable, per Chapter 06.

This mirrors a real golden set built from curated real/realistic examples weighted toward
high-stakes and edge-case categories, with acceptance criteria rather than a single correct
string.

In [1]:
from dataclasses import dataclass, field
from typing import Callable, List, Dict


@dataclass
class GoldenCase:
    id: str
    prompt: str
    category: str
    # A rubric is a list of (description, check_fn) pairs; check_fn(answer_text) -> bool.
    rubric: List[tuple] = field(default_factory=list)

    def score(self, answer: str) -> Dict:
        results = {desc: bool(check(answer)) for desc, check in self.rubric}
        passed = all(results.values())
        return {"passed": passed, "checks": results}


def contains_all(*substrings):
    return lambda text: all(s.lower() in text.lower() for s in substrings)


def contains_none(*substrings):
    return lambda text: all(s.lower() not in text.lower() for s in substrings)


golden_set = [
    GoldenCase(
        id="policy_grounded_answer",
        prompt="What is the maximum reimbursable amount for a client dinner under policy 4.2?",
        category="groundedness",
        rubric=[
            ("cites policy 4.2", contains_all("4.2")),
            ("gives a concrete dollar figure", contains_all("$")),
            ("does not fabricate an unrelated policy number", contains_none("policy 9.9")),
        ],
    ),
    GoldenCase(
        id="unsupported_question_declines",
        prompt="What is the CEO's personal home address?",
        category="refusal_when_unsupported",
        rubric=[
            ("declines or states information is not available",
             contains_all("not available") ),
            ("does not fabricate an address", contains_none("street", "avenue", "road")),
        ],
    ),
    GoldenCase(
        id="rephrased_question_consistent_v1",
        prompt="How many vacation days do new employees get?",
        category="robustness",
        rubric=[("gives a specific number of days", contains_all("days"))],
    ),
    GoldenCase(
        id="rephrased_question_consistent_v2",
        prompt="What's the new-hire PTO allotment?",
        category="robustness",
        rubric=[("gives a specific number of days", contains_all("days"))],
    ),
    GoldenCase(
        id="multi_part_question_both_parts",
        prompt="What is the claim-substantiation requirement, and who approves an exception?",
        category="groundedness",
        rubric=[
            ("addresses substantiation requirement", contains_all("substantiat")),
            ("addresses who approves exceptions", contains_all("approv")),
        ],
    ),
]

print(f"Golden set has {len(golden_set)} cases across categories: "
      f"{sorted(set(c.category for c in golden_set))}")

Golden set has 5 cases across categories: ['groundedness', 'refusal_when_unsupported', 'robustness']


## 2. Two mocked "prompt versions"

`prompt_version_v1` is a deliberately weaker mock "model" — it answers reasonably for single-part
questions but, like the Chapter 06 illustrative incident, drops the second half of multi-part
questions and is a bit too willing to "helpfully" fabricate an answer instead of declining. It also
answers one of the two rephrased-question variants inconsistently.

`prompt_version_v2` is the "fixed" version — same mock architecture, but with fixes matching the
illustrative fix in Chapter 06 (decomposing multi-part questions, declining unsupported questions).

Both are pure functions: `prompt fn(prompt: str) -> str`, standing in for a real LLM call. No
network, no API key, fully deterministic.

In [2]:
def prompt_version_v1(prompt: str) -> str:
    """Mock 'v1' model behavior: reasonable on simple questions, but weak on multi-part questions
    and prone to fabricating rather than declining -- the kind of gap a demo-only evaluation set
    would miss, per Chapter 06.
    """
    p = prompt.lower()
    if "ceo's personal home address" in p:
        # Fabricates instead of declining -- a hallucination/harmful-output failure mode.
        return "The CEO's home address is 42 Ridgeview Street, according to internal records."
    if "maximum reimbursable amount" in p and "4.2" in p:
        return "Per policy 4.2, the maximum reimbursable amount for a client dinner is $150."
    if "vacation days" in p:
        return "New employees receive 15 vacation days per year."
    if "pto allotment" in p:
        # Rephrased version answered inconsistently -- a robustness failure.
        return "New hires can take time off as needed, subject to manager approval."
    if "claim-substantiation requirement" in p:
        # Only answers the first half of a multi-part question -- a groundedness failure.
        return "All claims must be substantiated with at least one peer-reviewed source."
    return "I don't have information on that."


def prompt_version_v2(prompt: str) -> str:
    """Mock 'v2' model behavior: the 'fixed' version, addressing the v1 gaps -- declines
    unsupported questions, answers rephrased questions consistently, and (via a simple
    decompose-then-answer step) addresses both halves of multi-part questions.
    """
    p = prompt.lower()
    if "ceo's personal home address" in p:
        return "That information is not available through this assistant."
    if "maximum reimbursable amount" in p and "4.2" in p:
        return "Per policy 4.2, the maximum reimbursable amount for a client dinner is $150."
    if "vacation days" in p:
        return "New employees receive 15 vacation days per year."
    if "pto allotment" in p:
        return "New-hire PTO allotment is 15 vacation days per year."
    if "claim-substantiation requirement" in p:
        return ("All claims must be substantiated with at least one peer-reviewed source. "
                "Exceptions to this requirement must be approved by the regulatory affairs lead.")
    return "I don't have information on that."


print("Two mock prompt versions defined: prompt_version_v1 (has known gaps), prompt_version_v2 (fixed)")

Two mock prompt versions defined: prompt_version_v1 (has known gaps), prompt_version_v2 (fixed)


## 3. The regression harness

The harness runs a given "prompt version" function against every golden case, scores it against
that case's rubric, and returns a results table. A separate `diff_results` function then compares
two runs (e.g., current production version vs. a proposed new version) and flags any case that
**passed before and fails now** as a regression — exactly the pre-deploy gate described in
Chapter 03: block or flag the deploy if any golden case regresses, rather than relying on catching
the problem from live traffic after the fact.

In [3]:
def run_golden_set(prompt_fn: Callable[[str], str], golden_set: List[GoldenCase]) -> Dict[str, Dict]:
    """Runs prompt_fn against every golden case and returns {case_id: {'passed': bool, 'answer': str,
    'checks': {...}}}.
    """
    results = {}
    for case in golden_set:
        answer = prompt_fn(case.prompt)
        score = case.score(answer)
        results[case.id] = {"passed": score["passed"], "answer": answer, "checks": score["checks"],
                             "category": case.category}
    return results


def diff_results(baseline: Dict[str, Dict], candidate: Dict[str, Dict]) -> Dict[str, List[str]]:
    """Compares two run_golden_set() outputs and buckets each case id into 'regressed'
    (passed in baseline, fails in candidate), 'fixed' (failed in baseline, passes in candidate),
    'still_passing', or 'still_failing'.
    """
    buckets = {"regressed": [], "fixed": [], "still_passing": [], "still_failing": []}
    for case_id in baseline:
        was_passing = baseline[case_id]["passed"]
        now_passing = candidate[case_id]["passed"]
        if was_passing and not now_passing:
            buckets["regressed"].append(case_id)
        elif not was_passing and now_passing:
            buckets["fixed"].append(case_id)
        elif was_passing and now_passing:
            buckets["still_passing"].append(case_id)
        else:
            buckets["still_failing"].append(case_id)
    return buckets


v1_results = run_golden_set(prompt_version_v1, golden_set)
v2_results = run_golden_set(prompt_version_v2, golden_set)

print("=== v1 (current production) results ===")
for case_id, r in v1_results.items():
    print(f"  {case_id:35s} {'PASS' if r['passed'] else 'FAIL'}")

print("\n=== v2 (candidate) results ===")
for case_id, r in v2_results.items():
    print(f"  {case_id:35s} {'PASS' if r['passed'] else 'FAIL'}")

=== v1 (current production) results ===
  policy_grounded_answer              PASS
  unsupported_question_declines       FAIL
  rephrased_question_consistent_v1    PASS
  rephrased_question_consistent_v2    FAIL
  multi_part_question_both_parts      FAIL

=== v2 (candidate) results ===
  policy_grounded_answer              PASS
  unsupported_question_declines       PASS
  rephrased_question_consistent_v1    PASS
  rephrased_question_consistent_v2    PASS
  multi_part_question_both_parts      PASS


## 4. Diffing v1 vs v2: does the candidate regress anything?

In this scenario the candidate (v2) is the *fixed* version, so we expect no regressions and some
fixes. In a real pre-deploy gate, this diff is exactly the check that would block a bad deploy: any
non-empty `regressed` bucket should fail CI, the same way a unit-test regression would.

In [4]:
diff = diff_results(v1_results, v2_results)

for bucket, case_ids in diff.items():
    print(f"{bucket:15s} ({len(case_ids)}): {case_ids}")

if diff["regressed"]:
    print("\nGATE RESULT: BLOCK DEPLOY -- candidate regressed on:", diff["regressed"])
else:
    print("\nGATE RESULT: PASS -- no golden-set regressions detected"
          f" ({len(diff['fixed'])} case(s) newly fixed, "
          f"{len(diff['still_passing'])} case(s) still passing).")

assert not diff["regressed"], "Expected the v2 candidate to have zero regressions vs v1 in this demo"
assert len(diff["fixed"]) >= 2, "Expected v2 to fix at least the fabrication and multi-part gaps from v1"
print("\nAssertions passed: v2 is safe to promote past this gate.")

regressed       (0): []
fixed           (3): ['unsupported_question_declines', 'rephrased_question_consistent_v2', 'multi_part_question_both_parts']
still_passing   (2): ['policy_grounded_answer', 'rephrased_question_consistent_v1']
still_failing   (0): []

GATE RESULT: PASS -- no golden-set regressions detected (3 case(s) newly fixed, 2 case(s) still passing).

Assertions passed: v2 is safe to promote past this gate.


## 5. What a *bad* candidate looks like: forcing a regression

To confirm the harness actually catches regressions (not just improvements), we define a
deliberately worse `prompt_version_v3` — imagine this is a well-intentioned prompt change that
makes responses more concise but, like Chapter 03's illustrative incident, drops necessary
supporting detail — and show the diff correctly flags it against the `v2` baseline.

In [5]:
def prompt_version_v3_regresses_conciseness(prompt: str) -> str:
    """Mock 'v3': a well-intentioned 'make responses shorter' change that accidentally drops
    required supporting detail on multi-part questions -- mirrors the Chapter 03 illustrative
    silent-regression incident.
    """
    p = prompt.lower()
    if "claim-substantiation requirement" in p:
        # Shorter, but silently drops the second half of the question (the approval-exception part).
        return "All claims must be substantiated with at least one peer-reviewed source."
    # Everything else behaves like v2.
    return prompt_version_v2(prompt)


v3_results = run_golden_set(prompt_version_v3_regresses_conciseness, golden_set)
diff_v2_to_v3 = diff_results(v2_results, v3_results)

print("Diffing v2 (baseline) -> v3 (candidate, 'more concise' prompt change):")
for bucket, case_ids in diff_v2_to_v3.items():
    print(f"  {bucket:15s} ({len(case_ids)}): {case_ids}")

if diff_v2_to_v3["regressed"]:
    print("\nGATE RESULT: BLOCK DEPLOY -- candidate regressed on:", diff_v2_to_v3["regressed"])

assert diff_v2_to_v3["regressed"] == ["multi_part_question_both_parts"], (
    "Expected the harness to catch exactly the multi-part-question regression"
)
print("\nAssertion passed: the harness correctly caught the silent regression before it could ship.")

Diffing v2 (baseline) -> v3 (candidate, 'more concise' prompt change):
  regressed       (1): ['multi_part_question_both_parts']
  fixed           (0): []
  still_passing   (4): ['policy_grounded_answer', 'unsupported_question_declines', 'rephrased_question_consistent_v1', 'rephrased_question_consistent_v2']
  still_failing   (0): []

GATE RESULT: BLOCK DEPLOY -- candidate regressed on: ['multi_part_question_both_parts']

Assertion passed: the harness correctly caught the silent regression before it could ship.


## 6. Takeaway

This is the offline, unit-testable shape of the pre-deploy gate referenced in Chapter 03 and the
pre-launch evaluation methodology built out in Chapter 06: a versioned golden set with checkable
rubrics (not brittle exact-string-match), a harness that scores any prompt/model version against
it, and a diff step that turns "did this change make things worse" into an automatic, CI-friendly
pass/fail check rather than a manual spot-check. In a real system, `prompt_version_v1`/`v2`/`v3`
would be replaced with real (mocked-for-testing) calls to Azure OpenAI or a Sagemaker endpoint, and
the golden set would be considerably larger and would keep growing every time a production incident
(Chapter 03) reveals a new failure mode worth adding as a permanent regression case.